In [1]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
df = pd.read_csv("Market_Basket_Optimisation.csv", header = None);
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       7501 non-null   object
 1   1       5747 non-null   object
 2   2       4389 non-null   object
 3   3       3345 non-null   object
 4   4       2529 non-null   object
 5   5       1864 non-null   object
 6   6       1369 non-null   object
 7   7       981 non-null    object
 8   8       654 non-null    object
 9   9       395 non-null    object
 10  10      256 non-null    object
 11  11      154 non-null    object
 12  12      87 non-null     object
 13  13      47 non-null     object
 14  14      25 non-null     object
 15  15      8 non-null      object
 16  16      4 non-null      object
 17  17      4 non-null      object
 18  18      3 non-null      object
 19  19      1 non-null      object
dtypes: object(20)
memory usage: 1.1+ MB


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
transactions = []

for i in range(len(df)):
    transaction = []
    for j in range(len(df.columns)):
        item = str(df.iloc[i, j])
        if item != 'nan':
            transaction.append(item)
    transactions.append(transaction)

print(transactions[0:3])

[['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil'], ['burgers', 'meatballs', 'eggs'], ['chutney']]


In [4]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions) # we cannot give direct list of lists to mlxtend's apriori
                                                        # perform one-hot encoding on it

df_encoded = pd.DataFrame(te_array, columns=te.columns_)
df_encoded.head()

,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,True,True,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


In [5]:
frequent_itemsets = apriori(
    df_encoded,
    min_support=0.003,
    use_colnames=True
)

frequent_itemsets.head()

frequent_itemsets.sort_values(by="support", ascending=False).head(10)

,support,itemsets
69,0.238368,(mineral water)
34,0.179709,(eggs)
96,0.174110,(spaghetti)
40,0.170911,(french fries)
23,0.163845,(chocolate)
51,0.132116,(green tea)
68,0.129583,(milk)
52,0.098254,(ground beef)
46,0.095321,(frozen vegetables)
78,0.095054,(pancakes)


In [6]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(burgers),(almonds),0.087188,0.020397,0.005199,0.059633,2.923577,1.0,0.003421,1.041724,0.720799,0.050781,0.040053,0.157267
1,(almonds),(burgers),0.020397,0.087188,0.005199,0.254902,2.923577,1.0,0.003421,1.225089,0.671653,0.050781,0.183733,0.157267
2,(almonds),(cake),0.020397,0.081056,0.003066,0.150327,1.854607,1.0,0.001413,1.081527,0.470397,0.031165,0.075381,0.094078
3,(cake),(almonds),0.081056,0.020397,0.003066,0.037829,1.854607,1.0,0.001413,1.018117,0.501448,0.031165,0.017795,0.094078
4,(almonds),(chocolate),0.020397,0.163845,0.005999,0.294118,1.795099,1.0,0.002657,1.184553,0.452150,0.033657,0.155800,0.165366


In [7]:
rules[rules['confidence'] > 0.4]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
485,(cider),(eggs),0.010532,0.179709,0.004266,0.405063,2.253991,1.0,0.002373,1.378786,0.562264,0.022939,0.274724,0.214401
657,(extra dark chocolate),(mineral water),0.011998,0.238368,0.005733,0.477778,2.004369,1.0,0.002873,1.458444,0.507175,0.023433,0.314338,0.250913
985,(ground beef),(mineral water),0.098254,0.238368,0.040928,0.416554,1.747522,1.0,0.017507,1.305401,0.474369,0.138413,0.233952,0.294127
1105,(light cream),(mineral water),0.015598,0.238368,0.007332,0.470085,1.972098,1.0,0.003614,1.437273,0.500736,0.029730,0.304238,0.250423
1189,(nonfat milk),(mineral water),0.010399,0.238368,0.005066,0.487179,2.043811,1.0,0.002587,1.485182,0.516084,0.020788,0.326682,0.254216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4827,"(spaghetti, ground beef, tomatoes)",(mineral water),0.005599,0.238368,0.003066,0.547619,2.297366,1.0,0.001732,1.683607,0.567899,0.012728,0.406037,0.280241
4841,"(spaghetti, milk, olive oil)",(mineral water),0.007199,0.238368,0.003333,0.462963,1.942218,1.0,0.001617,1.418211,0.488642,0.013759,0.294886,0.238473
4855,"(spaghetti, milk, shrimp)",(mineral water),0.004933,0.238368,0.003066,0.621622,2.607821,1.0,0.001890,2.012884,0.619594,0.012764,0.503200,0.317243
4867,"(mineral water, milk, tomatoes)",(spaghetti),0.006532,0.174110,0.003333,0.510204,2.930353,1.0,0.002196,1.686192,0.663076,0.018797,0.406948,0.264673


In [8]:
rules[rules['lift'] > 3]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
90,(brownies),(cottage cheese),0.033729,0.031862,0.003466,0.102767,3.225330,1.0,0.002392,1.079026,0.714038,0.055794,0.073238,0.105777
91,(cottage cheese),(brownies),0.031862,0.033729,0.003466,0.108787,3.225330,1.0,0.002392,1.084220,0.712661,0.055794,0.077678,0.105777
364,(light cream),(chicken),0.015598,0.059992,0.004533,0.290598,4.843951,1.0,0.003597,1.325072,0.806131,0.063790,0.245324,0.183077
365,(chicken),(light cream),0.059992,0.015598,0.004533,0.075556,4.843951,1.0,0.003597,1.064858,0.844202,0.063790,0.060908,0.183077
644,(escalope),(mushroom cream sauce),0.079323,0.019064,0.005733,0.072269,3.790833,1.0,0.004220,1.057349,0.799635,0.061871,0.054239,0.186484
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4872,"(mineral water, tomatoes)","(spaghetti, milk)",0.024397,0.035462,0.003333,0.136612,3.852356,1.0,0.002468,1.117155,0.758934,0.058962,0.104869,0.115298
4873,"(spaghetti, milk)","(mineral water, tomatoes)",0.035462,0.024397,0.003333,0.093985,3.852356,1.0,0.002468,1.076807,0.767641,0.058962,0.071328,0.115298
4874,"(milk, tomatoes)","(mineral water, spaghetti)",0.013998,0.059725,0.003333,0.238095,3.986501,1.0,0.002497,1.234110,0.759789,0.047348,0.189700,0.146949
4875,"(spaghetti, tomatoes)","(mineral water, milk)",0.020931,0.047994,0.003333,0.159236,3.317852,1.0,0.002328,1.132311,0.713535,0.050813,0.116850,0.114340


In [9]:
rules[
    (rules['confidence'] > 0.4) &
    (rules['lift'] > 3)
].sort_values(by='lift', ascending=False)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4204,"(mineral water, whole wheat pasta)",(olive oil),0.009599,0.065858,0.003866,0.402778,6.115863,1.0,0.003234,1.564145,0.844598,0.054004,0.360673,0.230741
3935,"(spaghetti, tomato sauce)",(ground beef),0.006266,0.098254,0.003066,0.489362,4.980600,1.0,0.002451,1.765920,0.804260,0.030223,0.433723,0.260285
3220,"(herb & pepper, french fries)",(ground beef),0.006932,0.098254,0.003200,0.461538,4.697422,1.0,0.002518,1.674672,0.792612,0.031373,0.402868,0.247051
1921,"(spaghetti, cereals)",(ground beef),0.006666,0.098254,0.003066,0.460000,4.681764,1.0,0.002411,1.669901,0.791682,0.030105,0.401162,0.245604
4727,"(mineral water, frozen vegetables, soup)",(milk),0.005066,0.129583,0.003066,0.605263,4.670863,1.0,0.002410,2.205057,0.789908,0.023303,0.546497,0.314463
2452,"(herb & pepper, chocolate)",(ground beef),0.009065,0.098254,0.003999,0.441176,4.490183,1.0,0.003109,1.613652,0.784403,0.038710,0.380288,0.240941
4476,"(mineral water, shrimp, chocolate)",(frozen vegetables),0.007599,0.095321,0.003200,0.421053,4.417225,1.0,0.002475,1.562628,0.779537,0.032086,0.360052,0.227310
4713,"(mineral water, frozen vegetables, olive oil)",(milk),0.006532,0.129583,0.003333,0.510204,3.937285,1.0,0.002486,1.777102,0.750923,0.025100,0.437286,0.267962
1922,"(cereals, ground beef)",(spaghetti),0.004533,0.174110,0.003066,0.676471,3.885303,1.0,0.002277,2.552751,0.746001,0.017464,0.608266,0.347041
2066,"(chicken, olive oil)",(milk),0.007199,0.129583,0.003600,0.500000,3.858539,1.0,0.002667,1.740835,0.746207,0.027027,0.425563,0.263889
